# Identify change-related entity-property pairs

This notebook identifies change-related entity-property pairs in the 15,000 business sample.

Input file: `15000business_statements.xlsx`

Output files:  
`15000business_change_statements_candidates.xlsx`  
`15000business_non_change_statements.xlsx`  
`15000business_change_statements_final.xlsx`

The identification is conducted at the entity-property pair level. A pair is treated as change-related if it belongs to one of the following classes.

Class 1 contains directly change-expressing properties. These properties express factual change through the property-value structure itself.

Class 2 contains context-dependent properties. In the business domain, a pair is first selected as a candidate if at least one statement in the pair contains a selected change-related qualifier. These qualifiers provide a context about factual change to the property of the entity.

The automatic candidate identification is followed by manual validation. This validation removes candidate pairs where the selected change-related qualifier does not refer to the development history of the business entity, but instead describes such as source information, media files, or recurring temporal conditions

In [1]:
import pandas as pd

doc = pd.read_excel("15000business_statements.xlsx")

class1_prop_ids = {'P10786', 'P11812', 'P1317', 'P1319', 'P1326', 'P1365', 'P1366', 'P1398', 'P155','P156', 
                   'P1619', 'P167', 'P2031', 'P2032', 'P2669', 'P2754', 'P3999', 'P5204','P571', 'P575', 
                   'P576', 'P577', 'P580', 'P582', 'P585', 'P6949', 'P729', 'P730','P746', 'P7888', 'P807'}

change_qualifier_ids = {'P12413', 'P12506', 'P1264', 'P1319', 'P1326', 'P1365', 'P1366', 'P156', 'P2754', 'P3415',
                        'P571', 'P576', 'P577', 'P580', 'P582', 'P585', 'P7888', 'P8554', 'P8555'}


qualifier_id_cols = [
    column for column in doc.columns
    if column.startswith('Qualifier_Property_ID')
]

# Extract Class 1: Directly change-expressing properties
doc_class1 = doc[
    doc["Property_ID"].isin(class1_prop_ids)
].copy()

doc_class1["change_source"] = "direct_property"
doc_class1["change_detail"] = doc_class1["Property_ID"]
doc_class1['change_class'] = 'class1'

class1_pair_count = doc_class1.groupby(
    ["QID", "Property_ID"]
).ngroups

print(f"Class 1 pairs: {class1_pair_count}")


# Find Class 2 candidate statements
doc_class2_candidates = doc[
    ~doc["Property_ID"].isin(class1_prop_ids)
].copy()



def annotate_statement(row):
    triggered_quals = []
    for col in qualifier_id_cols:
        val = row[col]
        if pd.notna(val):
            qualifier_id = str(val).strip()
            if qualifier_id in change_qualifier_ids:
                triggered_quals.append(qualifier_id)
    if triggered_quals:
        return 'way1_qualifier', '|'.join(triggered_quals)
    
    return '', ''

annotations = doc_class2_candidates.apply(annotate_statement, axis=1)
doc_class2_candidates['change_source'] = annotations.apply(lambda x: x[0])
doc_class2_candidates['change_detail'] = annotations.apply(lambda x: x[1])

pair_has_change = doc_class2_candidates.groupby(['QID', 'Property_ID'])['change_source'].apply(
    lambda x: any(v != '' for v in x)
).reset_index()
pair_has_change.columns = ['QID', 'Property_ID', 'has_change']

c2_pairs = pair_has_change[
    pair_has_change['has_change']
][['QID', 'Property_ID']]


doc_class2 = doc_class2_candidates.merge(
    c2_pairs, on=['QID', 'Property_ID'], how='inner'
)
doc_class2['change_class'] = 'class2'
print(f"Class 2 pairs:{doc_class2.groupby(['QID','Property_ID']).ngroups}")

# Merge pair of class 1 and class 2
doc_class1['change_class'] = 'class1'
doc_change = pd.concat([doc_class1, doc_class2], ignore_index=True)
doc_change = doc_change.sort_values(
    by=['QID', 'Property_ID', 'Statement_Number']
).reset_index(drop=True)

print(f"\nChange-related total: {len(doc_change)} statements")
print(f"\nChange-related pairs: {doc_change.groupby(['QID','Property_ID']).ngroups}")


# Non-change
change_keys = doc_change[
    ['QID', 'Property_ID', 'Statement_Number']
].drop_duplicates()
change_keys['_in_change'] = True

doc_non_change = doc.merge(
    change_keys, on=['QID', 'Property_ID', 'Statement_Number'], how='left'
)
doc_non_change = doc_non_change[
    doc_non_change['_in_change'].isna()
].drop(columns=['_in_change'])

doc_non_change = doc_non_change.sort_values(
    by=['QID', 'Property_ID', 'Statement_Number']
).reset_index(drop=True)


print(f"Non-change-related pairs: {doc_non_change.groupby(['QID','Property_ID']).ngroups}")

doc_change.to_excel('15000business_change_statements_candidates.xlsx', index=False)
doc_non_change.to_excel('15000business_non_change_statements.xlsx', index=False)

print("\nSaved.")

Class 1 pairs: 7342
Class 2 pairs:5628

Change-related total: 17475 statements

Change-related pairs: 12970
Non-change-related pairs: 118699

Saved.
